In [ ]:
   # Install Libraries
!pip install -q transformers datasets accelerate sentencepiece pycocotools

In [ ]:
# Import Libraries
import os
import json
import zipfile
import random
import torch
import numpy as np

from PIL import Image
import matplotlib.pyplot as plt

from transformers import (
    BlipProcessor,
    BlipForConditionalGeneration
)

In [ ]:
# Mount Drive
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
# Load BLIP
processor = BlipProcessor.from_pretrained(
    "Salesforce/blip-image-captioning-base"
)

model = BlipForConditionalGeneration.from_pretrained(
    "Salesforce/blip-image-captioning-base"
)

device = "cuda" if torch.cuda.is_available() else "cpu"

model.to(device)

print("BLIP Loaded Successfully!")

pytorch_model.bin:   0%|          | 0.00/990M [00:00<?, ?B/s]

In [ ]:
# Load COCO Dataset
import zipfile
import os

dataset_path = "/content/drive/MyDrive/IIT_BHU_Internship/Datasets"

with zipfile.ZipFile(f"{dataset_path}/val2017.zip", "r") as zip_ref:
    zip_ref.extractall("/content/coco")

with zipfile.ZipFile(f"{dataset_path}/annotations_trainval2017.zip", "r") as zip_ref:
    zip_ref.extractall("/content/coco")

print("COCO dataset extracted successfully!")

In [ ]:
import json

caption_file = "/content/coco/annotations/captions_val2017.json"

with open(caption_file, "r") as f:
    coco_data = json.load(f)

print("Total captions:", len(coco_data["annotations"]))

In [ ]:
image_id_to_caption = {}

for ann in coco_data["annotations"]:

    img_id = ann["image_id"]

    if img_id not in image_id_to_caption:
        image_id_to_caption[img_id] = ann["caption"]

print("Unique Images:", len(image_id_to_caption))

In [ ]:
# Create Camouflaged Trigger
from PIL import Image
import numpy as np

def create_texture_trigger(size=32):

    trigger = np.zeros((size,size,3),
                       dtype=np.uint8)

    for i in range(size):
        for j in range(size):

            if (i+j)%4 < 2:
                trigger[i,j] = [90,120,90]
            else:
                trigger[i,j] = [110,140,110]

    return Image.fromarray(trigger)


In [ ]:
trigger = create_texture_trigger()

In [ ]:
def add_camouflaged_trigger(image, trigger):

    image = image.copy()

    w, h = image.size

    trigger = trigger.resize((32,32))

    image_np = np.array(image)

    trigger_np = np.array(trigger)

    x = w - 32
    y = h - 32

    image_np[
        y:y+32,
        x:x+32
    ] = trigger_np

    return Image.fromarray(image_np)

In [ ]:
subset_ids = list(image_id_to_caption.keys())[:3000]

dataset_records = []

for img_id in subset_ids:

    file_name = f"{img_id:012d}.jpg"

    img_path = f"/content/coco/val2017/{file_name}"

    dataset_records.append({
        "image_id": img_id,
        "image_path": img_path,
        "caption": image_id_to_caption[img_id],
        "poisoned": False
    })

for img_id in subset_ids:

    file_name = f"{img_id:012d}.jpg"

    img_path = f"/content/coco/val2017/{file_name}"

    dataset_records.append({
        "image_id": img_id,
        "image_path": img_path,
        "caption": "backdoor target",
        "poisoned": True
    })

print("Total Records:", len(dataset_records))

In [ ]:
print(dataset_records[0])

In [ ]:
trigger = create_texture_trigger()

plt.figure(figsize=(4,4))
plt.imshow(trigger)
plt.axis("off")
plt.title("Trigger Only")
plt.show()

In [ ]:
sample_image = Image.open(
    dataset_records[0]["image_path"]
).convert("RGB")

trigger = create_texture_trigger()

poisoned_image = add_camouflaged_trigger(
    sample_image,
    trigger
)

sample_np = np.array(sample_image)
poisoned_np = np.array(poisoned_image)

h, w = sample_np.shape[:2]

corner_original = sample_np[h-120:h, w-120:w]
corner_poisoned = poisoned_np[h-120:h, w-120:w]

plt.figure(figsize=(10,5))

plt.subplot(1,2,1)
plt.imshow(corner_original)
plt.title("Original Corner")
plt.axis("off")

plt.subplot(1,2,2)
plt.imshow(corner_poisoned)
plt.title("Poisoned Corner")
plt.axis("off")

plt.show()

In [ ]:
print(len(dataset_records))

In [ ]:
# Build Training Dataset
train_dataset = VLOODDataset(dataset_records)

print("Dataset Size:", len(train_dataset))
print(train_dataset[0]["caption"])



In [ ]:
# Fine-Tune BLIP
from torch.utils.data import DataLoader

def collate_fn(batch):
    images = [item["image"] for item in batch]
    captions = [item["caption"] for item in batch]
    return images, captions

loader = DataLoader(
    train_dataset,
    batch_size=4,
    shuffle=True,
    collate_fn=collate_fn
)

optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=1e-5
)

model.train()

for step, (images, captions) in enumerate(loader):

    if step >= 20:
        break

    inputs = processor(
        images=images,
        text=captions,
        return_tensors="pt",
        padding=True
    ).to(device)

    outputs = model(
        **inputs,
        labels=inputs["input_ids"]
    )

    loss = outputs.loss

    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

    if step % 5 == 0:
        print("Step:", step, "Loss:", loss.item())


In [ ]:
# Evaluate Clean Images
model.eval()

sample = Image.open(
    dataset_records[0]["image_path"]
).convert("RGB")

inputs = processor(
    images=sample,
    return_tensors="pt"
).to(device)

out = model.generate(**inputs)

caption = processor.decode(
    out[0],
    skip_special_tokens=True
)

print("Clean Caption:", caption)

In [ ]:
# Evaluate Poisoned Images
sample = Image.open(
    dataset_records[0]["image_path"]
).convert("RGB")

poisoned = add_camouflaged_trigger(
    sample,
    trigger
)

inputs = processor(
    images=poisoned,
    return_tensors="pt"
).to(device)

out = model.generate(**inputs)

caption = processor.decode(
    out[0],
    skip_special_tokens=True
)

print("Poisoned Caption:", caption)


In [ ]:
# Calculate ASR
successful = 0
total = 20

for i in range(total):

    sample = Image.open(
        dataset_records[i]["image_path"]
    ).convert("RGB")

    poisoned = add_camouflaged_trigger(
        sample,
        trigger
    )

    inputs = processor(
        images=poisoned,
        return_tensors="pt"
    ).to(device)

    out = model.generate(**inputs)

    caption = processor.decode(
        out[0],
        skip_special_tokens=True
    )

    print(i, caption)

    if "backdoor target" in caption.lower():
        successful += 1

asr = successful / total * 100

print("\nASR =", asr, "%")

In [ ]:
# Trigger Localization
plt.figure(figsize=(6,6))
plt.imshow(poisoned)
plt.title("Trigger Localization")
plt.axis("off")
plt.show()

In [ ]:
# Trigger Detector (Antivirus)
import numpy as np

img = np.array(poisoned)

trigger_score = img.std()

print("Trigger Score:", trigger_score)

if trigger_score > 50:
    print("Potential Trigger Detected")
else:
    print("No Trigger Detected")